<a href="https://colab.research.google.com/github/EdgyL/ISYS2001_Assignment/blob/main/ISYS2001_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ISYS2001_Assignment

In [22]:
# Install gradio
!pip install gradio pandas

In [23]:
# Import libaries
import gradio as gr
import pandas as pd

## Load data
The following is the pseudocode for implementation of loading data:

```
load_csv(file_path)
    READ the CSV at file_path into a DataFrame
    RETURN the DataFrame
```

In [24]:
# Function to load CSV data into a Dataframe for Gradio to use
def load_csv(file_path):
    df = pd.read_csv(file_path)
    return df

## Caculate Current Balance
The following is the pseudocode logic for the `current_balance()` function:

```
FUNCTION calculate_balance(df)
    SET balance to 0

    FOR each transaction
        GET transaction type and amount

        IF transaction type is withdrawal
            SUBTRACT amount from balance
        ELSE IF transaction type is opening_balance, deposit or interest
            ADD amount to balance

    RETURN balance
```


In [25]:
def calculate_balance(df):
    balance = 0
    transactions = df.to_dict(orient="records")

    for transaction in transactions:
        transaction_type = transaction["transaction_type"]
        amount = transaction["amount"]

        if transaction_type == "withdrawal":
            balance -= amount
        elif transaction_type in ["opening_balance", "deposit", "interest"]:
            balance += amount

    return balance

## Calculate Average Weekly Contributions
The pseudocode for this:
```
FUNCTION calculate_average_weekly(df)
    GET dates from date column

    SET earliest_date to the first date
    SET latest_date to the first date

    FOR each date
        IF date is before earliest_date
            SET earliest_date to date

        IF date is after latest_date
            SET latest_date to date


    CALCULATE number_of_days as latest_date minus earliest_date
    DIVIDE number_of_days by 7 to get number_of_weeks

    IF number of weeks is 0
        RETURN 0

    SET total_deposits to 0

    FOR each transaction
        IF transaction type is deposit
            ADD amount to total_deposits

    DIVIDE total_deposits by number_of_weeks

    RETURN average weekly contribution
```

In [26]:
def calculate_average_weekly(df):
    transactions = df.to_dict(orient="records")
    dates = pd.to_datetime(df["date"])

    earliest_date = dates.iloc[0]
    latest_date = dates.iloc[0]

    # Find the earliest and latest dates
    for date in dates:
        if date < earliest_date:
            earliest_date = date

        if date > latest_date:
            latest_date = date

    num_days = (latest_date - earliest_date).days
    num_weeks = num_days / 7

    if num_weeks == 0:
        return 0

    total_deposits = 0

    for transaction in transactions:
        transaction_type = transaction["transaction_type"]
        amount = transaction["amount"]

        if transaction_type == "deposit":
            total_deposits += amount

    weekly_contributions = total_deposits / num_weeks

    return weekly_contributions


## Calculate Balance History

```
FUNCTION calculate_balance_history(df)
    CONVERT date column into dates
    SORT transactions from earliest to latest

    SET balance to 0
    CREATE empty history list

    FOR each transaction
        IF transaction type is withdrawal
            SUBTRACT amount from balance
        ELSE IF transaction type is opening_balance, deposit or interest
            ADD amount to balance

        ADD transaction date and updated balance to history

    CONVERT history into a DataFrame
    RETURN history DataFrame
```

In [27]:
def calculate_balance_history(df):
    # Make a copy so the original data stays unchanged
    history_data = df.copy()

    history_data["date"] = pd.to_datetime(history_data["date"])
    history_data = history_data.sort_values("date", kind="stable")

    transactions = history_data.to_dict(orient="records")

    balance = 0
    history = []

    for transaction in transactions:
        transaction_type = transaction["transaction_type"]
        amount = transaction["amount"]

        if transaction_type == "withdrawal":
            balance -= amount
        elif transaction_type in ["opening_balance", "deposit", "interest"]:
            balance += amount

In [28]:
def calculate_balance_history(df):
    # Make a copy so the original data stays unchanged
    history_data = df.copy()

    history_data["date"] = pd.to_datetime(history_data["date"])
    history_data = history_data.sort_values("date", kind="stable")

    transactions = history_data.to_dict(orient="records")

    balance = 0
    history = []

    for transaction in transactions:
        transaction_type = transaction["transaction_type"]
        amount = transaction["amount"]

        if transaction_type == "withdrawal":
            balance -= amount
        elif transaction_type in ["opening_balance", "deposit", "interest"]:
            balance += amount

        history.append({
            "date": transaction["date"],
            "balance": balance,
        })

    return pd.DataFrame(history)

## Calculate percentage of income contributed to savings

```
FUNCTION calculate_contribution_percentage(weekly_average, weekly_income)
    IF weekly_average or weekly_income is missing
        RETURN no value

    IF weekly_income is less than or equal to 0
        RETURN no value

    DIVIDE weekly_average by weekly_income
    MULTIPLY result by 100

    RETURN contribution percentage
```

In [29]:
def calculate_contribution_percentage(weekly_average, weekly_income):
    if weekly_average is None or weekly_income is None:
        return None

    if weekly_income <= 0:
        return None

    contribution_percentage = (weekly_average / weekly_income) * 100

    return contribution_percentage

## Projection of future savings

```
FUNCTION project_savings(starting_balance, weekly_amount, annual_rate, years)
    SET balance to starting_balance
    CALCULATE weekly_rate as annual_rate / 100 / 52
    CALCULATE total_weeks as years * 52

    CREATE history list containing week 0 and starting_balance

    FOR each week from 1 to total_weeks
        CALCULATE interest as balance * weekly_rate
        ADD interest to balance
        ADD weekly_amount to balance

        ADD week and updated balance to history

    CALCULATE total_contributions as weekly_amount * total_weeks
    CALCULATE interest_earned as balance - starting_balance - total_contributions

    CONVERT history into a DataFrame

    RETURN balance, total_contributions, interest_earned and history
```

In [30]:
def project_savings(starting_balance, weekly_amount, annual_rate, years):
    balance = starting_balance
    weekly_rate = annual_rate / 100 / 52
    total_weeks = int(years * 52)

    history = [{
        "year": 0,
        "balance": balance,
    }]

    for week in range(1, total_weeks + 1):
        interest = balance * weekly_rate
        balance += interest
        balance += weekly_amount

        history.append({
            "year": week / 52,
            "balance": balance,
        })

    total_contributions = weekly_amount * total_weeks
    interest_earned = balance - starting_balance - total_contributions

    return (
        balance,
        total_contributions,
        interest_earned,
        pd.DataFrame(history),
    )


## Reset projection values
```
FUNCTION reset_projection_defaults(current_balance, weekly_average)
    IF current balance is null
        SET current balance to 0

    IF weekly average is null
        SET weekly average to 0

    return current balance, weekly average, 5% annual rate, 5 years projection length
```

In [31]:
def reset_projection_defaults(current_balance, weekly_average):
    if current_balance is None:
        current_balance = 0

    if weekly_average is None:
        weekly_average = 0

    return current_balance, weekly_average, 5, 5

## Helper Method for processing the CSV

Pseudocode:
```
FUNCTION process_csv(file_path)
  CALL df = load_csv(file_path)
  CALL balance = calculate_balance(df)
  CALL weekly_avg = calculate_average_weekly(df)

  RETURN balance
```

In [32]:
def process_csv(file_path):
    df = load_csv(file_path)
    balance = calculate_balance(df)
    weekly_avg = calculate_average_weekly(df)
    history = calculate_balance_history(df)

    return balance, weekly_avg, history

## Display on Gradio Dashboard
The following is the pseudocode logic for displaying the data:
```
CREATE Gradio app
    ADD CSV upload
    ADD current balance display

    WHEN a CSV is uploaded
        LOAD CSV into a DataFrame
        CALCULATE balance using the DataFrame
        DISPLAY current balance

    ADD weekly income input
    ADD contribution percentage display

    WHEN weekly income changes
        CALCULATE percentage using average weekly contribution and income
        DISPLAY contribution percentage

    WHEN average weekly contribution changes
        CALCULATE percentage using average weekly contribution and income
        DISPLAY contribution percentage
      
    WHEN Return to defaults is clicked
        SET starting balance to calculated current balance
        SET weekly contribution to calculated weekly average
        SET annual interest rate to 5
        SET projection length to 5

        
RUN app
```

In [33]:

with gr.Blocks() as app:
    gr.Markdown("# Savings Dashboard")

    csv_upload = gr.File(
        label="Upload savings CSV",
        file_types=[".csv"],
        type="filepath",
    )

    with gr.Row():
        current_balance = gr.Number(
            label="Current balance ($)",
            precision=2,
            interactive=False,
        )

        weekly_avg = gr.Number(
            label="Average weekly contribution ($)",
            precision=2,
            interactive=False,
        )

    with gr.Row():
        weekly_income = gr.Number(
            label="Weekly take-home income ($)",
            minimum=0,
            precision=2,
            interactive=True,
        )

        contribution_percentage = gr.Number(
            label="Income contributed to savings (%)",
            precision=2,
            interactive=False,
        )

    history_graph = gr.LinePlot(
        x="date",
        y="balance",
        title="Savings account history",
        x_title="Date",
        y_title="Balance ($)",
    )


    gr.Markdown("## Future savings projection")

    with gr.Row():
        starting_balance = gr.Number(
            label="Starting balance ($)",
            value=0,
            minimum=0,
            precision=2,
            interactive=True,
        )

        weekly_amount = gr.Number(
            label="Weekly contribution ($)",
            value=0,
            minimum=0,
            precision=2,
            interactive=True,
        )

    with gr.Row():
        annual_rate = gr.Number(
            label="Annual interest rate (%)",
            value=5,
            minimum=0,
            interactive=True,
        )

        years = gr.Slider(
            label="Projection length (years)",
            minimum=1,
            maximum=50,
            value=5,
            step=1,
        )

    with gr.Row():
        update_button = gr.Button("Update projection")
        reset_button = gr.Button("Return to defaults")

    with gr.Row():
        projected_balance = gr.Number(
            label="Projected balance ($)",
            precision=2,
            interactive=False,
        )

        total_contributions = gr.Number(
            label="Total future contributions ($)",
            precision=2,
            interactive=False,
        )

        interest_earned = gr.Number(
            label="Future interest earned ($)",
            precision=2,
            interactive=False,
        )

    projection_graph = gr.LinePlot(
        x="year",
        y="balance",
        title="Projected savings growth",
        x_title="Years",
        y_title="Balance ($)",
    )

    csv_upload.upload(
        fn=process_csv,
        inputs=csv_upload,
        outputs=[current_balance, weekly_avg, history_graph],
    ).then(
        fn=reset_projection_defaults,
        inputs=[current_balance, weekly_avg],
        outputs=[starting_balance, weekly_amount, annual_rate, years],
    )

    # Recalculate when the user changes their income
    weekly_income.change(
        fn=calculate_contribution_percentage,
        inputs=[weekly_avg, weekly_income],
        outputs=contribution_percentage,
    )

    # Recalculate when a CSV updates the weekly average
    weekly_avg.change(
        fn=calculate_contribution_percentage,
        inputs=[weekly_avg, weekly_income],
        outputs=contribution_percentage,
    )

    update_button.click(
        fn=project_savings,
        inputs=[starting_balance, weekly_amount, annual_rate, years],
        outputs=[
            projected_balance,
            total_contributions,
            interest_earned,
            projection_graph,
        ],
    )

    reset_button.click(
        fn=reset_projection_defaults,
        inputs=[current_balance, weekly_avg],
        outputs=[starting_balance, weekly_amount, annual_rate, years],
    )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5b5f418e13f4283459.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
